<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/diff_bir_super_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
import os, shutil

if os.path.isdir('/content/DiffBIR'):
    shutil.rmtree('/content/DiffBIR')

!git clone --depth 1 https://github.com/XPixelGroup/DiffBIR.git
%cd DiffBIR

# requirements.txt репозитория жёстко пинит torch==2.2.2+cu118 / xformers под конкретную
# сборку CUDA — это ломает уже установленный в Colab torch. Убираем torch/torchvision/
# torchaudio/xformers из списка и используем то, что уже стоит в среде Colab.
# bitsandbytes нужен только для 4-бит квантования LLaVA-captioner'а — тоже убираем,
# т.к. по умолчанию ниже используется captioner=none (без LLaVA).
!grep -vE '^(--extra-index-url|torch==|torchvision==|torchaudio==|xformers==|bitsandbytes==)' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q huggingface_hub

import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

# --- Patches ------------------------------------------------------------------
# В коде сэмплеров (diffbir/sampler/*.py) есть опечатка: аннотации типов вида
# `torch.Tuple[...]` вместо `Tuple[...]`. На старых версиях torch это иногда
# проходило незамеченным, а на актуальном torch (2.11+, как в свежем Colab)
# атрибута `torch.Tuple` не существует, и импорт diffbir.inference падает с
# `AttributeError: module 'torch' has no attribute 'Tuple'` ещё до старта инференса.
# Убираем лишний префикс `torch.`, а затем ГАРАНТИРОВАННО добавляем
# `from typing import ...` в начало каждого затронутого файла — сами эти
# имена в файле заранее не импортированы (это и есть причина краша:
# исходно `torch.Tuple` без импорта -> AttributeError, просто срезав
# префикс `torch.` получили бы NameError). Повторный/лишний импорт typing
# ничего не ломает, даже если имя уже импортировано выше.
import subprocess
patched_files = subprocess.run(
    ['grep', '-rlE', r'torch\.(Tuple|List|Dict|Optional|Union)\[', 'diffbir/'],
    capture_output=True, text=True
).stdout.split()
for fpath in patched_files:
    !sed -i -E 's/torch\.(Tuple|List|Dict|Optional|Union)\[/\1[/g' {fpath}
    !sed -i '1i from typing import Any, Dict, List, Optional, Tuple, Union' {fpath}
print('Патчи применены к файлам:', patched_files or '(ничего не потребовалось)')

# --- Веса -------------------------------------------------------------------
# В отличие от SeeSR, веса DiffBIR (IRControlNet, SwinIR/BSRNet/SCUNet) скачиваются
# автоматически при первом запуске inference.py (см. diffbir/inference/
# pretrained_models.py) и лежат в открытых репозиториях lxq007/... — с ними проблем нет.
#
# А вот веса базовой Stable Diffusion 2.1 (v2-1_512-ema-pruned.ckpt) в коде указаны
# как stabilityai/stable-diffusion-2-1-base, а Stability AI официально удалили
# репозитории SD 2.0/2.1 с Hugging Face (депрекация моделей, консолидация линейки
# + требования EU AI Act 2026) — поэтому оригинальный URL отдаёт `404 Repository
# Not Found` (и токен тут не поможет, репозитория просто больше нет).
# Качаем тот же самый файл (SHA256 совпадает с оригиналом) с открытого зеркала
# ckpt/stable-diffusion-2-1-base — токен не нужен.
from huggingface_hub import hf_hub_download

os.makedirs('weights', exist_ok=True)
sd21_target = os.path.join('weights', 'v2-1_512-ema-pruned.ckpt')

if not os.path.exists(sd21_target):
    print('Скачиваю v2-1_512-ema-pruned.ckpt с зеркала ckpt/stable-diffusion-2-1-base...')
    downloaded_path = hf_hub_download(
        repo_id='ckpt/stable-diffusion-2-1-base',
        filename='v2-1_512-ema-pruned.ckpt',
    )
    shutil.copy(downloaded_path, sd21_target)
    print('Готово:', sd21_target)
else:
    print('Файл уже на месте, повторно не скачиваю:', sd21_target)

print('DiffBIR склонирован, зависимости установлены.')

# --- RAM-патч: OOM при загрузке чекпоинтов -------------------------------------
# torch.load() полностью материализует чекпоинт SD 2.1 (fp32, ~5.2 GB) в RAM
# ОДНОВРЕМЕННО с уже собранной fp32-архитектурой (SD-бэкбон + ControlNet),
# а сами sd_weight/control_weight (fp32, ~5.2 GB и ~2.5 GB) потом ещё и не
# удаляются, пока не закончится вся функция load_cldm(). Суммарно легко
# набегает 10-11+ GB, и ядро Colab (лимит ~12.7 GB) убивает процесс
# OOM-killer'ом (код -9) прямо во время сборки/загрузки весов, до первого
# шага сэмплирования.
path = 'diffbir/utils/common.py'
src = open(path, encoding='utf-8').read()
old = 'sd = torch.load(sd_path, map_location="cpu")'
# weights_only=False обязателен: v2-1_512-ema-pruned.ckpt — старый формат
# PyTorch Lightning, внутри которого кроме тензоров лежит объект
# pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint. Начиная с
# torch 2.6 дефолт weights_only=True запрещает распаковывать такие
# нетензорные классы и падает с UnpicklingError. Источник (официальное
# зеркало на HF) доверенный, поэтому явный weights_only=False безопасен.
new = 'sd = torch.load(sd_path, map_location="cpu", mmap=True, weights_only=False)'
assert old in src, 'RAM-патч #1: строка не найдена — репозиторий обновился, нужна ручная правка'
open(path, 'w', encoding='utf-8').write(src.replace(old, new))
print('common.py: mmap=True + weights_only=False включены в torch.load')

path = 'diffbir/inference/loop.py'
src = open(path, encoding='utf-8').read()

old1 = '''        unused, missing = self.cldm.load_pretrained_sd(sd_weight)
        print(
            f"load pretrained stable diffusion, "
            f"unused weights: {unused}, missing weights: {missing}"
        )'''
new1 = old1 + '\n        del sd_weight\n        gc.collect()'
assert old1 in src, 'RAM-патч #2a: строка не найдена'
src = src.replace(old1, new1)

old2 = '''        self.cldm.load_controlnet_from_ckpt(control_weight)
        print(f"load controlnet weight")
        self.cldm.eval().to(self.args.device)'''
new2 = '''        self.cldm.load_controlnet_from_ckpt(control_weight)
        print(f"load controlnet weight")
        del control_weight
        gc.collect()
        self.cldm.eval().to(self.args.device)'''
assert old2 in src, 'RAM-патч #2b: строка не найдена'
src = src.replace(old2, new2)

if 'import gc' not in src:
    src = src.replace('import os\n', 'import os\nimport gc\n', 1)

open(path, 'w', encoding='utf-8').write(src)
print('loop.py: sd_weight/control_weight освобождаются сразу после копирования в модель')
print('RAM-патч применён.')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os
import shutil
from google.colab import files

upload_folder = './inputs/user_upload'
result_folder = './results/user_upload'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)

os.makedirs(upload_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# Upload images — you can select multiple files at once
uploaded = files.upload()
for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moved {filename} -> {dst_path}')
    shutil.move(filename, dst_path)

print(f'\n{len(uploaded)} image(s) uploaded to {upload_folder}')

In [ ]:
#@title ##**Run** { display-mode: "form" }
import os, re, gc, codecs, torch, psutil, subprocess, sys, time, threading

start_time_main = time.time()

# --- Prevent working directory loss after Restart session --------------
if os.path.basename(os.getcwd()) != 'DiffBIR':
    if os.path.isdir('/content/DiffBIR'):
        os.chdir('/content/DiffBIR')
        print(f'⚠️ Working directory was reset. Switched to {os.getcwd()}')
    else:
        raise RuntimeError('Folder /content/DiffBIR not found. Please run the "Install" cell.')

# --- Prevent upload_folder/result_folder variables loss ---------------
if 'upload_folder' not in globals():
    upload_folder = './inputs/user_upload'
    print(f'⚠️ upload_folder was not defined. Using: {upload_folder}')
if 'result_folder' not in globals():
    result_folder = './results/user_upload'
    print(f'⚠️ result_folder was not defined. Using: {result_folder}')

if not os.path.isdir(upload_folder) or not os.listdir(upload_folder):
    raise RuntimeError(f'No images found in {upload_folder}. Please run "Upload Images".')
os.makedirs(result_folder, exist_ok=True)

task = "sr" #@param ["sr", "face", "face_background", "denoise"]
version = "v2" #@param ["v1", "v2", "v2.1"]
upscale = 3 #@param {type:"slider", min:1, max:8, step:1}
steps = 50 #@param {type:"slider", min:5, max:100, step:1}
sampler = "spaced" #@param ["spaced", "ddim", "dpm++_m2", "edm_dpm++_3m_sde", "edm_dpm++_2m"]
captioner = "none" #@param ["none", "ram", "llava"]
cfg_scale = 4.0 #@param {type:"number"}
noise_aug = 0 #@param {type:"slider", min:0, max:200, step:10}
pos_prompt = "Hyperealism" #@param {type:"string"}
neg_prompt = "blur" #@param {type:"string"}
seed = 231 #@param {type:"integer"}
low_vram_tiled_sampling = True #@param {type:"boolean"}
show_log = False #@param {type:"boolean"}

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
print(f'Free RAM: {psutil.virtual_memory().available/1024**3:.2f} GiB')

IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff')
total_images = len([f for f in os.listdir(upload_folder) if f.lower().endswith(IMAGE_EXTS)])
if total_images == 0:
    total_images = 1

cmd = [
    'python', '-u', 'inference.py',
    '--task', task,
    '--upscale', str(upscale),
    '--version', version,
    '--sampler', sampler,
    '--steps', str(steps),
    '--captioner', captioner,
    '--pos_prompt', pos_prompt,
    '--neg_prompt', neg_prompt,
    '--cfg_scale', str(cfg_scale),
    '--noise_aug', str(noise_aug),
    '--seed', str(seed),
    '--input', upload_folder,
    '--output', result_folder,
    '--device', 'cuda', '--precision', 'fp16',
]
if low_vram_tiled_sampling:
    cmd += [
        '--cleaner_tiled', '--cleaner_tile_size', '256', '--cleaner_tile_stride', '128',
        '--vae_encoder_tiled', '--vae_encoder_tile_size', '256',
        '--vae_decoder_tiled', '--vae_decoder_tile_size', '256',
        '--cldm_tiled', '--cldm_tile_size', '512', '--cldm_tile_stride', '256',
    ]

last_output_time = [time.time()]
stop_heartbeat = threading.Event()

def heartbeat():
    while not stop_heartbeat.is_set():
        time.sleep(15)
        idle = time.time() - last_output_time[0]
        if idle > 14 and not stop_heartbeat.is_set():
            msg = f'... process is alive, running silently for {idle:.0f} sec (loading weights) ...'
            sys.stdout.write('\r' + ' ' * 110 + '\r' + msg + '\n')
            sys.stdout.flush()

hb_thread = threading.Thread(target=heartbeat, daemon=True)
hb_thread.start()

# --- Output parsing ---------------------------------------------------------
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
buffer = ''

images_done = 0
current_percent = 0
ansi_escape = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')

main_bar_text = ""
sub_bar_text = ""
last_was_bar = False

def render_compact_status():
    bar_len = 20
    filled = int(bar_len * current_percent / 100)
    bar = '█' * filled + '░' * (bar_len - filled)
    sys.stdout.write(f'\rProcessing images: {images_done}/{total_images}  [{bar}] {current_percent:3d}%   ')
    sys.stdout.flush()

def handle_segment(segment, sep):
    global images_done, current_percent, main_bar_text, sub_bar_text, last_was_bar

    clean_seg = ansi_escape.sub('', segment).strip()
    if not clean_seg:
        return

    if 'save result to' in clean_seg:
        images_done += 1

    m = re.search(r'(\d+)%\|', clean_seg)
    if m and 'Tiled' not in clean_seg and 'Queue' not in clean_seg:
        current_percent = int(m.group(1))

    if not show_log:
        render_compact_status()
        return

    is_bar = '%|' in clean_seg or 'it/s' in clean_seg or 's/it' in clean_seg

    if is_bar:
        if bool(re.match(r'^\d+%\|', clean_seg)):
            main_bar_text = clean_seg
            sub_bar_text = ""
        else:
            sub_bar_text = clean_seg

        combined = f"{main_bar_text}   {sub_bar_text}".strip()
        out_line = combined[:110].ljust(110)

        sys.stdout.write('\r' + out_line)
        sys.stdout.flush()
        last_was_bar = True
    else:
        if last_was_bar:
            sys.stdout.write('\r' + ' ' * 110 + '\r')

        sys.stdout.write(clean_seg + '\n')

        if last_was_bar:
            combined = f"{main_bar_text}   {sub_bar_text}".strip()
            sys.stdout.write(combined[:110].ljust(110))

        sys.stdout.flush()

try:
    while True:
        chunk = process.stdout.read(4096)
        if not chunk:
            break
        last_output_time[0] = time.time()
        buffer += decoder.decode(chunk)
        while True:
            idx_r = buffer.find('\r')
            idx_n = buffer.find('\n')
            if idx_r == -1 and idx_n == -1:
                break
            if idx_n == -1 or (idx_r != -1 and idx_r < idx_n):
                segment, sep, buffer = buffer[:idx_r], '\r', buffer[idx_r + 1:]
            else:
                segment, sep, buffer = buffer[:idx_n], '\n', buffer[idx_n + 1:]
            handle_segment(segment, sep)

    buffer += decoder.decode(b'', final=True)
    if buffer:
        handle_segment(buffer, '\n')
    process.wait()
finally:
    stop_heartbeat.set()
    print()

if not show_log:
    current_percent = 100
    render_compact_status()
    print()

if process.returncode != 0:
    print(f'\n⚠️ inference.py exited with code {process.returncode}')
else:
    print(f'\n✅ Done! Processed images -> {result_folder}')

# --- Calculate and print elapsed time ---------------------------------------
elapsed_time = time.time() - start_time_main
m, s = divmod(elapsed_time, 60)
h, m = divmod(m, 60)

time_str = ""
if h > 0:
    time_str += f"{int(h)}h "
if m > 0 or h > 0:
    time_str += f"{int(m)}m "
time_str += f"{s:.1f}s"

print(f'⏱️ Total execution time: {time_str}')
print(f'🖥️ RAM available after run: {psutil.virtual_memory().available/1024**3:.2f} GiB')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os
import base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# inference.py DiffBIR обычно кладёт результаты прямо в result_folder (без вложенных
# sampleNN/, как у SeeSR), но на случай изменения структуры репозитория ищем
# картинки рекурсивно.
def find_images(root):
    found = []
    for dirpath, _, fnames in os.walk(root):
        for fname in sorted(fnames):
            if is_image_file(fname):
                found.append(os.path.join(dirpath, fname))
    return sorted(found)

filenames_input = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)])
paths_output = find_images(result_folder)

if not filenames_input or not paths_output:
    print(f'Error: no images found in {upload_folder} or {result_folder}.')
else:
    for filename, path_output in zip(filenames_input, paths_output):
        try:
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(path_output)

            max_width = min(image_after.size[0], 1000)
            image_after = resize_image_maintain_aspect(image_after, max_width)
            target_height = image_after.size[1]
            image_before = resize_image_maintain_aspect(image_before, max_width, target_height)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            html_code = f"""
            <div style="position: relative; width: {image_after.size[0]}px; height: {image_after.size[1]}px; margin-bottom: 20px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <img src="data:image/png;base64,{after_base64}" style="position: absolute; width: 100%; height: 100%;">
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>
            <script>
                document.querySelectorAll('.slider').forEach(slider => {{
                    let isDragging = false;
                    const container = slider.parentElement.querySelector('div:nth-child(1)');
                    const clipDiv = container.querySelector('div');

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }});
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {path_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os
from google.colab import files

zip_filename = 'DiffBIR_result.zip'
if os.path.exists(zip_filename):
    os.remove(zip_filename)

os.system(f"zip -r -j {zip_filename} {result_folder}/*")
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>